# Plotting inference results
This notebook loads and plots the following inference results:
1. Overview of summary statistics over networks and decades
2. Inference results
    - Distances by model variant
    - Contour plot of posterior estimates
3. Predictive analysis using best performing models

In [ ]:
from typing import Dict, Tuple, Optional
import os
from collections import defaultdict

import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from netin.models import CompoundLFM

from patch.constants import (
    PATH_INFERENCE, MAP_LFM_SHORT, SIZE_FIG,
    MAP_MODEL_COLOR, MAP_LFM_SHORT,
    APS, DBLP, APS_CIT,
    PATH_PLOTS, MAP_DATA_COLOR,
    MAP_STAT_LABEL_SHORT, N_SAMPLES,
    PATH_INFERENCE_PREDICTIVE)
from patch.statistics import compute_contour_lines

In [ ]:
# Set figure size
plt.rcParams["figure.figsize"] = SIZE_FIG
# Remove legend border
plt.rcParams["legend.frameon"] = False
# Remove top and right axis
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# Set font size
plt.rcParams["font.size"] = 10

In [ ]:
PATH_PREFIX = "08-04_gini-sep_" # Prefix to identify inference run

LFM_COMBS = [
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value),
    (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.PAH.value, CompoundLFM.PAH.value),
]

# Define decades for different datasets
L_DECADES_APS = [1970, 1980, 1990, 2000, 2010]
L_DECADES_DBLP = [1970, 1980, 1990, 2000]
L_DECADES_APS_CIT = [1970, 1980, 1990, 2000, 2010]

N_BINS = 15
DURATION = 10 # Decade duration
PERCENTILE_THRESHOLDS = [.85, .5, .15] # For contour plots

In [ ]:
def create_folder_name(
        source: str, decade: int,
        lfm_global: str, lfm_tc: str,
        prefix: str = PATH_PREFIX) -> str:
    """Create folder name based on source, decade, and LFM types.

    This function constructs a folder name that includes the source of the
    data, the decade of interest, and the types of latent factor models
    (LFM) used for global and temporal coupling.

    The folder name is structured as follows:
    `../inference/{source}/{prefix}lfm-g-{lfm_global}_lfm-t-{lfm_tc}_d-{decade}/`

    Parameters
    ----------
    source : str
        The source of the data, e.g., "aps", "dblp", or "aps_cit".
    decade : int
        The decade of interest, e.g., 1970, 1980, etc.
    lfm_global : str
        The type of global link formation mechanism.
    lfm_tc : str
        The type of triadic closure link formation mechanism.
    prefix : str, optional
        The prefix for the folder name, by default PATH_PREFIX

    Returns
    -------
    str
        The constructed folder name.
    """
    return os.path.join(
        "..",
        PATH_INFERENCE,
        source,
        f"{prefix}lfm-g-{lfm_global}_lfm-t-{lfm_tc}_d-{decade}/")


In [ ]:
def create_posterior_file_name(folder_name: str) -> str:
    """Create the file name for the posterior results.

    Parameters
    ----------
    folder_name : str
        The name of the folder where the posterior results are stored.

    Returns
    -------
    str
        The constructed file name for the posterior results.
    """
    return os.path.join(folder_name, "posteriors.npz")

In [ ]:
folder_plots = os.path.join("../", PATH_PLOTS, PATH_PREFIX, "empirical")
if not os.path.exists(folder_plots):
    os.makedirs(folder_plots)
print(f"Saving plots to {folder_plots}")

Load sample result file.

In [ ]:
np_file = np.load(
    create_posterior_file_name(
        create_folder_name(
            source=APS_CIT,
            lfm_global=CompoundLFM.PAH.value,
            lfm_tc=CompoundLFM.UNIFORM.value,
            decade=2010)))
print(np_file.files)

## Inequalities overview

In [ ]:
def plot_fm(
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
    is_subplot: bool = True)\
        -> Tuple[plt.Figure, plt.Axes]:
    """Plot the fraction of minority nodes over decades.

    Parameters
    ----------
    tpl_fig_ax : Optional[Tuple[plt.Figure, plt.Axes]], optional
        A tuple containing a matplotlib Figure and Axes object to plot on.
        If None, a new Figure and Axes will be created.
    is_subplot : bool, optional
        If True, the plot will be created as a subplot. Defaults to True.

    Returns
    -------
    Tuple[plt.Figure, plt.Axes]
        A tuple containing the Figure and Axes objects of the plot.
    """
    fig, ax = tpl_fig_ax if tpl_fig_ax is not None else plt.subplots()
    l_fm_aps, l_fm_dblp, l_fm_aps_cit = [], [], []
    for source, l_decades, l_fm in zip(
            [APS, DBLP, APS_CIT],
            [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT],
            [l_fm_aps, l_fm_dblp, l_fm_aps_cit]):
        for decade in l_decades:
            np_file = np.load(
                create_posterior_file_name(
                    create_folder_name(
                        source=source,
                        lfm_global=CompoundLFM.HOMOPHILY.value,
                        lfm_tc=CompoundLFM.HOMOPHILY.value,
                        decade=decade,
                        prefix=PATH_PREFIX)))
            l_fm.append(np_file['f_m'])

    for l_decades, l_fm, source in zip(
            [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT],
            [l_fm_aps, l_fm_dblp, l_fm_aps_cit],
            [APS, DBLP, APS_CIT]):
        ax.plot(
            l_decades,
            l_fm,
            marker="o",
            color=MAP_DATA_COLOR[source],
            label=str.upper(source))

    # Format y-tick labels to percentage
    ax.set_yticks(
        [.05, .1],
        labels=["{:.0%}".format(x)\
                for x in [.05, .1]])
    ax.set_ylabel(
        "$f_m$",
        # Bring closer to axis
        labelpad=-5)

    if not is_subplot:
        ax.set_xlabel(
            "Decade",
            # Bring closer to axis
            labelpad=-.5
        )
        _ = ax.legend()
    return fig, ax

In [ ]:
def collect_observed_statistic(
    statistic: str,
    lfm_global: str = CompoundLFM.HOMOPHILY.value,
    lfm_tc: str = CompoundLFM.HOMOPHILY.value,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Collects observed statistics for different datasets.

    Parameters
    ----------
    statistic : str
        The name of the statistic to collect, e.g., "gini".
    lfm_global : str, optional
        The global link formation mechanism to use, by default CompoundLFM.HOMOPHILY.value
    lfm_tc : str, optional
        The triadic closure link formation mechanism to use, by default CompoundLFM.HOMOPHILY.value

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        The observed statistics for each dataset.
    """
    a_stats_aps = np.zeros(len(L_DECADES_APS))
    a_stats_dblp = np.zeros(len(L_DECADES_DBLP))
    a_stats_aps_cit = np.zeros(len(L_DECADES_APS_CIT))
    for source, l_decades, a_stats in zip(
        [APS, DBLP, APS_CIT],
        [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT],
        [a_stats_aps, a_stats_dblp, a_stats_aps_cit]):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    a_stats[j] = data[statistic]
    return a_stats_aps, a_stats_dblp, a_stats_aps_cit

In [ ]:
def transform_observed_statistic(
        statistic: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Transforms the observed statistics.

    Transformations:
    - For "ei", rescales from [0, 1] to [-1, +1].

    Parameters
    ----------
    statistic : str
        The name of the statistic to transform, e.g., "gini".

    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        The transformed statistics for each dataset.
    Returns
    -------
    Tuple[np.ndarray, np.ndarray, np.ndarray]
        The transformed statistics for APS, DBLP, and APS_CIT datasets.
    """
    a_stats_aps, a_stats_dblp, a_stats_aps_cit = collect_observed_statistic(statistic)
    if statistic == "ei":
        # Rescale from [0, 1] to [-1, +1]
        a_stats_aps = 2 * a_stats_aps - 1
        a_stats_dblp = 2 * a_stats_dblp - 1
        a_stats_aps_cit = 2 * a_stats_aps_cit - 1
    return a_stats_aps, a_stats_dblp, a_stats_aps_cit

In [ ]:
def plot_summary_stat_over_decades(
        statistic: str,
        vneutral: Optional[float] = None,
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
        is_subplot: bool = True,
        labels: bool = True)\
            -> Tuple[plt.Figure, plt.Axes]:
    """Plot summary statistics over decades.
    """
    fig, ax = tpl_fig_ax if tpl_fig_ax is not None else plt.subplots()

    a_stats_aps, a_stats_dblp, a_stats_aps_cit = transform_observed_statistic(statistic)

    if vneutral is not None:
        ax.axhline(
            vneutral,
            color='gray',
            linestyle=':')

    for source, l_decades, a_stats in zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT],
            [a_stats_dblp, a_stats_aps, a_stats_aps_cit]):
        ax.plot(
            l_decades,
            a_stats,
            marker="o",
            color=MAP_DATA_COLOR[source],
            label=str.upper(source) if labels else None)

    ax.text(
        .5, 1.1,
        MAP_STAT_LABEL_SHORT[statistic],
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes)
    ax.set_xticks(
        [L_DECADES_APS[0], L_DECADES_APS[-1]],
        labels=[
            f"{L_DECADES_APS[0]}",
            f"'{L_DECADES_APS[-1] % 100}"])

    if not is_subplot:
        _ = ax.legend()

    return fig, ax

Plot the evolution of inequality metrics.

In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]/2.75),
    layout='constrained')
gs = fig.add_gridspec(1, 6)

# Plot fraction of minority nodes
ax_fm = fig.add_subplot(gs[0, 0])
plot_fm(tpl_fig_ax=(fig, ax_fm))
ax_fm.set_ylabel('')
ax_fm.text(
    .5, 1.1,
    r"$f_{\mathregular{min}}$",
    horizontalalignment='center',
    verticalalignment='center',
    transform=ax_fm.transAxes)

# Plot summary statistics
a_ax = [fig.add_subplot(gs[0, i])\
        for i in range(1, 6)]
for i, (statistic, vneutral) in enumerate(zip(
        ["ei", "gini_min", "gini_maj", "mann_whitney", "ccf"],
        [0., None, None, 0.5, None])):
    plot_summary_stat_over_decades(
        statistic,
        vneutral=vneutral,
        tpl_fig_ax=(fig, a_ax[i],),
        labels=False)

ax_fm.set_yticks(
    [.01, .20],
    labels=["1%", "20%"])
ax_fm.set_ylim(.01, .20)

a_ax[0].set_yticks(
    [-1, 0,])

a_ax[1].set_yticks(
    [.3, .6])
a_ax[1].set_ylim(.3, .6)

a_ax[2].set_yticks(
    [.3, .6])
a_ax[2].set_ylim(.3, .6)

a_ax[3].set_yticks(
    [.4, .5])
a_ax[3].set_ylim(.4, .51)

a_ax[4].set_yticks(
    [.15, .55])
a_ax[4].set_ylim(top=.55, bottom=.15)

for i, ax in enumerate([ax_fm] + a_ax):
    ax.margins(y=.1)
    ax.set_xticks(
        [L_DECADES_APS[0], L_DECADES_APS[-1]],
        labels=[
            "'70" if i != 0 else "1970",
            "'10" if i != 0 else "2010"])
    ax.set_xticks(
        L_DECADES_APS[1:-1],
        minor=True)
    # Add space
    ax.set_xlim(1970 - 5, 2010 + 5)

for letter, ax in zip(
        ["a", "b", "c", "d", "e", "f"],
        [ax_fm] + a_ax):
    ax.text(
        -.125, 1.15,
        letter,
        fontweight='bold',
        horizontalalignment='center',
        verticalalignment='center',
        transform=ax.transAxes)

fig.legend(
    ncol=3,
    handlelength=1.,
    bbox_to_anchor=(0.5, 1.05),
    loc='center',
)
fig.supxlabel("Decade", fontsize=10, y=-.075)

fig.savefig(os.path.join(
    folder_plots, "summary_stats.pdf"), bbox_inches='tight')

## Model selection

In [ ]:
def load_discrepancies(
    source: str,
    lfm_global: str,
    lfm_tc: str,
    decade: int,
    prefix: str = PATH_PREFIX
) -> np.ndarray:
    """Load discrepancies from a posterior file.
    """
    file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade,
                    prefix=prefix))
    with np.load(file_name) as data:
        return data['discrepancies']

In [ ]:
def get_closest_lfm(
    source: str,
    decade: int
) -> Tuple[str, str]:
    """Return the closest LFM combination based on the Mann-Whitney U test.

    This function computes the Mann-Whitney U statistic for each combination of
    latent factor models (LFM) and returns the one with the smallest value.
    It compares the discrepancies of a candidate LFM combination against all
    other combinations.

    Parameters
    ----------
    source : str
        The source dataset (e.g., "aps", "dblp", "aps_cit").
    decade : int
        The decade to consider (e.g., 2020).

    Returns
    -------
    Tuple[str, str]
        The closest LFM combination (global, triadic closure).
    """
    res_mw = []
    for lfm_g_candidate, lfm_t_candidate in LFM_COMBS:
        disc_cand = load_discrepancies(
            source=source,
            lfm_global=lfm_g_candidate,
            lfm_tc=lfm_t_candidate,
            decade=decade,
            prefix=PATH_PREFIX
        )
        disc_cmp = []
        for lfm_global, lfm_tc in LFM_COMBS:
                if (lfm_global, lfm_tc) == (lfm_g_candidate, lfm_t_candidate):
                    continue
                disc = load_discrepancies(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade,
                    prefix=PATH_PREFIX
                )
                disc_cmp = np.concatenate((disc_cmp, disc))
        res_mw.append(
             sc.stats.mannwhitneyu(
                 disc_cand, disc_cmp,
                 alternative="less").statistic / (len(disc_cand) * len(disc_cmp))
        )
    return LFM_COMBS[np.argmin(res_mw)]


Get median and 2.5/97.5-quantiles to for later plot and verbose output.

In [ ]:
post_aps_quant = {combo: [] for combo in LFM_COMBS}
post_dblp_quant = {combo: [] for combo in LFM_COMBS}
post_aps_cit_quant = {combo: [] for combo in LFM_COMBS}
for source, l_decades, results in zip(
        [APS, DBLP, APS_CIT],
        [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT],
        [post_aps_quant, post_dblp_quant, post_aps_cit_quant]):
    for decade in l_decades:
        for lfm_global, lfm_tc in LFM_COMBS:
            results[(lfm_global, lfm_tc)].append(
                    np.quantile(load_discrepancies(
                        source=source,
                        lfm_global=lfm_global,
                        lfm_tc=lfm_tc,
                        decade=decade,
                        prefix=PATH_PREFIX
                    ), q=(0.025, 0.5, 0.975)))

Select best performing model.

In [ ]:
MODEL_SELECTION = defaultdict(dict)

print("Model selection results:")
for source, a_quantiles, a_decades in zip(
    [APS, DBLP, APS_CIT],
    [post_aps_quant, post_dblp_quant, post_aps_cit_quant],
    [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT]):
    print(f"Source: {source}")
    for i, decade in enumerate(a_decades):
        lfm_global, lfm_tc = get_closest_lfm(
            source=source,
            decade=decade
        )
        MODEL_SELECTION[source][decade] = (lfm_global, lfm_tc)
        print(f"  {decade}: {lfm_global}, {lfm_tc}")
        print("  Means: " + ", ".join([
            f"{lfm_global},{lfm_tc}: "\
                +f"{a_quantiles[(lfm_global, lfm_tc)][i][1]:.2f}"\
                    for lfm_global, lfm_tc in LFM_COMBS
        ]))


## Posterior analysis

In [ ]:
def _plot_single_source_decade_bars(
    ax: plt.Axes,
    i_decade: int,
    j_model: int,
    decade: int,
    lfm_global: str,
    lfm_tc: str,
    put_labels: bool = False
):
    is_wo_dblp = i_decade == len(L_DECADES_DBLP)
    vals_dblp = None if is_wo_dblp else\
        np.array(post_dblp_quant[(lfm_global, lfm_tc)][i_decade])
    vals_aps = np.array(post_aps_quant[(lfm_global, lfm_tc)][i_decade])
    vals_aps_cit = np.array(post_aps_cit_quant[(lfm_global, lfm_tc)][i_decade])

    if not is_wo_dblp:
        ax.bar(
            [j_model - .3, j_model, j_model + .3],
            [vals_dblp[1], vals_aps[1], vals_aps_cit[1]],
            width=.3,
            yerr=[[
                vals_dblp[1] - vals_dblp[0],
                vals_aps[1] - vals_aps[0],
                vals_aps_cit[1] - vals_aps_cit[0]],[
                vals_dblp[2] - vals_dblp[1],
                vals_aps[2] - vals_aps[1],
                vals_aps_cit[2] - vals_aps_cit[1]]],
            color=[
                MAP_DATA_COLOR[DBLP],
                MAP_DATA_COLOR[APS],
                MAP_DATA_COLOR[APS_CIT]],
            label=[str.upper(source) for source in (DBLP, APS, APS_CIT)]\
                if put_labels else None)
    else:
        ax.bar(
            [j_model, j_model + .3],
            [vals_aps[1], vals_aps_cit[1]],
            width=.3,
            yerr=[[
                vals_aps[1] - vals_aps[0],
                vals_aps_cit[1] - vals_aps_cit[0]],[
                vals_aps[2] - vals_aps[1],
                vals_aps_cit[2] - vals_aps_cit[1]]],
            color=[
                MAP_DATA_COLOR[APS],
                MAP_DATA_COLOR[APS_CIT]])

    for source, x_off, vals in zip(
            (DBLP, APS, APS_CIT) if not is_wo_dblp else (APS, APS_CIT),
            (-.3, 0, .3) if not is_wo_dblp else (0, .3),
            (vals_dblp, vals_aps, vals_aps_cit) if not is_wo_dblp else (vals_aps, vals_aps_cit)):
        if (lfm_global, lfm_tc) == MODEL_SELECTION[source][decade]:
            ax.text(
                j_model + x_off,
                vals[1] + 3,
                "*",
                color=MAP_DATA_COLOR[source],
                fontweight='bold',
                verticalalignment='center',
                horizontalalignment='center',
                fontsize=8)

In [ ]:
def plot_model_selection_bar(
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,)\
        -> Tuple[plt.Figure, plt.Axes]:
    # Plotting
    l_artists = []
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(ncols=len(L_DECADES_APS), sharey=True, sharex=True)
    for j, (lfm_global, lfm_tc) in enumerate(LFM_COMBS):
        for i, decade in enumerate(L_DECADES_DBLP):
            _plot_single_source_decade_bars(
                ax=a_ax[i],
                i_decade=i,
                j_model=j,
                decade=decade,
                lfm_global=lfm_global,
                lfm_tc=lfm_tc,
                put_labels=(i == 0 and j == 0))
        _plot_single_source_decade_bars(
            ax=a_ax[-1],
            i_decade=len(L_DECADES_APS) - 1,
            j_model=j,
            decade=L_DECADES_APS[-1],
            lfm_global=lfm_global,
            lfm_tc=lfm_tc
        )

    a_ax[0].set_xticks(
        range(len(LFM_COMBS)),
        labels=[f"{MAP_LFM_SHORT[lfm_global]}\n{MAP_LFM_SHORT[lfm_tc]}"\
                for lfm_global, lfm_tc in LFM_COMBS])

    for ax in a_ax[1:]:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', which='both', labelleft=False, left=False)

    a_ax[0].set_ylabel('Distance', labelpad=-.5)

    # Print data set names on top of last bars in first subplot
    # for i, (xoff, source, vals) in enumerate(zip(
    #         (-.3, 0.0, .3),
    #         # (0.05, .35, .55),
    #         (DBLP, APS, APS_CIT),
    #         (post_dblp_quant, post_aps_quant, post_aps_cit_quant))):
    #     post_dblp_quant[(lfm_global, lfm_tc)]
    #     a_ax[0].text(
    #         len(LFM_COMBS) - 1 + xoff - 1 + (i*0.035),
    #         vals[LFM_COMBS[-2]][0][1] + 3,
    #         str.upper(source),
    #         # transform=ax.transAxes,
    #         rotation=90,
    #         fontsize=8,
    #         color=MAP_DATA_COLOR[source],
    #         verticalalignment='bottom',
    #         horizontalalignment='center')

    # Print decades on top
    for ax, decade in zip(a_ax, L_DECADES_APS):
        ax.text(
            0.5, 1.1,
            s=f"{decade}" + u"\u2014" + f"{decade + DURATION}",
            horizontalalignment='center',
            verticalalignment='bottom',
            transform=ax.transAxes,
            fontsize=10)

    return fig, a_ax, l_artists

In [ ]:
def plot_posterior_contour(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
        is_subplot: bool = True) \
        -> Tuple[plt.Figure, plt.Axes]:
    # Plotting
    fig, a_ax = tpl_fig_ax \
        if tpl_fig_ax is not None else \
            plt.subplots(
                ncols=len(L_DECADES_APS),
                sharex=True,
                sharey=True)

    a_cntr_h, a_cntr_tau = np.zeros(
        (2, 3, len(L_DECADES_APS), N_SAMPLES))
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            lfm_global, lfm_tc = MODEL_SELECTION[source][decade]
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    a_cntr_h[i, j] = data['h']
                    a_cntr_tau[i, j] = data['tau']

    l_artists = []
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            if j >= len(a_ax):
                continue

            X, Y, Z, thresholds = compute_contour_lines(
                a_tau=a_cntr_tau[i, j],
                a_h=a_cntr_h[i, j],
                percentiles=PERCENTILE_THRESHOLDS,
            )

            # Create the contour plot
            for threshold, alpha in zip(thresholds, (.5, .75, 1.)):
                contour = a_ax[j].contour(
                    X, Y, Z,
                    levels=[threshold],
                    colors=[MAP_DATA_COLOR[source]],
                    alpha=alpha,
                )
                l_artists.append(contour)

    if not is_subplot and len(l_artists) > 0:
        a_ax[0].legend()

    # Set axis limits and labels
    for ax in a_ax:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
        ax.set_xlabel(
            "$\\tau$",
            labelpad=-.5)
        ax.set_xticks(
            [0, .5, 1],
            labels=["0", "", "1"])

    a_ax[0].set_ylabel(
        "$h$",
        # Bring closer to axis
        labelpad=-.5)
    a_ax[0].set_yticks(
        [0, .5, 1],
        labels=["0", "", "1"])

    for ax in a_ax[1:]:
        ax.tick_params(axis='y', labelleft=False)

    # Add clabel for first plot only
    for contour, pct, alpha, (x,y) in zip(
            l_artists[27:30], PERCENTILE_THRESHOLDS, (.5, .75, 1.), ((.99, .9), (.99, .1), (.7, .1))):
        # a_ax[0].clabel(
        #     contour,
        #     inline=True,
        #     inline_spacing=2.5,
        #     fontsize=8,
        #     fmt=lambda _: f"{pct:.0%}")
        a_ax[0].text(
            x,
            y,
            f"{pct:.0%}",
            alpha=alpha,
            fontsize=8,
            verticalalignment='center',
            horizontalalignment='right',
            color=MAP_DATA_COLOR[APS_CIT],)

    return fig, a_ax, l_artists

In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1] * (2/3)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=2,
    height_ratios=[1, 1.5]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=len(L_DECADES_APS))

# Plot model selection
a_ax_model_select = [
    fig.add_subplot(
        gs_model_select[0])]
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_model_select.append(
        fig.add_subplot(
            gs_model_select[i],
            sharex=a_ax_model_select[0],
            sharey=a_ax_model_select[0]))
_ = plot_model_selection_bar(
    tpl_fig_ax=(fig, a_ax_model_select))

# DBLP
gs_posteriors = gs[1].subgridspec(
    nrows=1,
    ncols=1+len(L_DECADES_APS),
    width_ratios=[0.05, 1, 1, 1, 1, 1])

# Plot posterior samples
a_ax_posteriors = [
    fig.add_subplot(gs_posteriors[1])]
for i, decade in enumerate(L_DECADES_APS[1:], start=2):
    a_ax_posteriors.append(fig.add_subplot(
        gs_posteriors[i],
        sharey=a_ax_posteriors[0],
        sharex=a_ax_posteriors[0]))
a_ax_posteriors = np.asarray(a_ax_posteriors)

_ = plot_posterior_contour(
    tpl_fig_ax=(
        fig, a_ax_posteriors))

fig.text(0.02, 0.96, "a", fontweight='bold')
fig.text(0.02, 0.6, "b", fontweight='bold')
fig.text(0.045, .68, "$\\mathregular{{L_G}}\colon$")
fig.text(0.045, .6275, "$\\mathregular{{L_T}}\colon$")

fig.legend(
    ncol=3,
    handlelength=.85,
    bbox_to_anchor=(0.5, 1.025),
    loc='center',
    fontsize=10
)

fig.savefig(os.path.join(
    folder_plots, "inference_gauss.pdf"), bbox_inches='tight')


## Predictive Check

In [ ]:
L_METRICS = ["ccf", "ei", "gini_min", "gini_maj", "mann_whitney", "gini_comp"]

In [ ]:
def create_pred_folder_name(
        source: str, decade: int,
        lfm_global: str, lfm_tc: str,
        prefix: str = PATH_PREFIX) -> str:
    return os.path.join(
        "..",
        PATH_INFERENCE_PREDICTIVE,
        source,
        f"{prefix}lfm-g-{lfm_global}_lfm-t-{lfm_tc}_d-{decade}/")

def create_pred_file_name(
        folder_path: str
) -> str:
    return os.path.join(
        folder_path,
        "predictive.npz")

In [ ]:
def _plot_predictive(
        metric_values_pred: np.ndarray,
        metric_obs: float,
        source: str,
        ax: plt.Axes
):
    ax.hist(
        metric_values_pred,
        bins=50,
        density=True,
        color=MAP_DATA_COLOR[source],
        alpha=.75)
    ax.axvline(
        metric_obs,
        color="black")

In [ ]:
def plot_pred_analysis_per_metric(
        metric: str
):
    f, a_ax = plt.subplots(
        ncols=len(L_DECADES_APS),
        constrained_layout=True,
        nrows=3,)

    for source, a_ax_src, l_decades in zip(
            [DBLP, APS, APS_CIT],
            a_ax,
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT]):
        for i, (decade, ax) in enumerate(zip(l_decades, a_ax_src)):
            lfm_global, lfm_tc = MODEL_SELECTION[source][decade]
            file_name_pred = create_pred_file_name(
                create_pred_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade,
                    prefix=PATH_PREFIX))
            metric_values_pred = None
            with np.load(file_name_pred) as data:
                metric_values_pred = data[metric]\
                    if metric != "gini_comp" else data["gini_min"] / data["gini_maj"]

            file_name_post = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            metric_obs = None
            with np.load(file_name_post) as data:
                metric_obs = data[metric]\
                    if metric != "gini_comp" else data["gini_min"] / data["gini_maj"]

            _plot_predictive(
                metric_values_pred=metric_values_pred,
                metric_obs=metric_obs,
                source=source,
                ax=ax
            )
            ax.text(
                0.05, 0.05,
                f"{MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}",
                horizontalalignment='left',
                verticalalignment='bottom',
                transform=ax.transAxes,
                fontsize=8,
                # white background
                bbox=dict(
                    facecolor='white',
                    edgecolor='none',
                    alpha=0.75,
                    boxstyle='round,pad=0.25'
                ))
    for ax in a_ax[:, 0]:
        ax.set_ylabel(f"$P(${MAP_STAT_LABEL_SHORT[metric]}$)$", labelpad=-.5)
    for ax, decade in zip((a_ax[0, :]),
                        L_DECADES_APS):
        ax.set_title(f"{decade}", fontsize=10, pad=-.5)
    a_ax[-1, 2].set_xlabel(f"{MAP_STAT_LABEL_SHORT[metric]}")
    for source, pos_y in zip(
            [DBLP, APS, APS_CIT],
            np.linspace(1., 0.375, 3)):
        f.text(
            0.0, pos_y,
            source.upper(),
            fontweight='bold',
            horizontalalignment='left',
            verticalalignment='top',)
    return f, a_ax


In [ ]:
for metric in L_METRICS:
    f, _ = plot_pred_analysis_per_metric(metric)
    f.savefig(
        os.path.join(
            folder_plots,
            f"pred_analysis_{metric}.pdf"
    ))

## Appendix

### Alternative plotting

In [ ]:
fig, a_ax = plt.subplots(
    ncols=len(L_DECADES_APS), nrows=3, sharex="col", layout='constrained',)

for source, a_decades, ax_source in zip(
    [APS, DBLP, APS_CIT], [L_DECADES_APS, L_DECADES_DBLP, L_DECADES_APS_CIT], a_ax):
    for decade, ax_decade in zip(
        a_decades, ax_source
    ):
        a_disc = np.zeros((len(LFM_COMBS), 1000))
        for i, (lfm_global, lfm_tc) in enumerate(LFM_COMBS):
            a_disc[i] = load_discrepancies(
                source=source,
                lfm_global=lfm_global,
                lfm_tc=lfm_tc,
                decade=decade,
                prefix=PATH_PREFIX
            )
        a_bins = np.linspace(
            np.min(a_disc),
            np.max(a_disc),
            N_BINS + 1
        )
        for i, (lfm_global, lfm_tc) in enumerate(LFM_COMBS):
            ax_decade.hist(
                a_disc[i],
                bins=a_bins,
                density=True,
                label=f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$",
                color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
                alpha=0.5)
        ax_decade.set_title(
            f"{source} {decade}", fontsize=9)
a_ax[0, 0].legend()

In [ ]:
def plot_model_selection(
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
    is_subplot: bool = True)\
        -> Tuple[plt.Figure, plt.Axes]:
    # Plotting
    l_artists = []
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(ncols=3, sharey=True, sharex=True)
    for i, (results, l_decades) in enumerate(zip(
            [post_dblp_quant, post_aps_quant, post_aps_cit_quant],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for (lfm_global, lfm_tc), values in results.items():
            values = np.array(values)
            artist = a_ax[i].errorbar(
                l_decades,
                values[:, 1],
                yerr=[values[:, 1] - values[:, 0], values[:, 2] - values[:, 1]],
                fmt='o-',
                color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
                label=f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$" if i == 0 else None)
            l_artists.append(artist)

            if i == 0:
                # Plot model labels to the right of last markers
                y_off = 1.25\
                    if (lfm_global, lfm_tc) == (CompoundLFM.PAH.value, CompoundLFM.PAH.value)\
                        else -1.25\
                            if (lfm_global, lfm_tc) == (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value)\
                                else 0
                a_ax[i].text(
                    L_DECADES_DBLP[-1] + 1.5,
                    values[-1, 1] + y_off,
                    f"{MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}",
                    color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
                    verticalalignment='center',
                    transform=a_ax[i].transData,
                    # Bold if PAH, U
                    fontweight='bold'\
                        if ((lfm_global == CompoundLFM.PAH.value) and (lfm_tc == CompoundLFM.UNIFORM.value))\
                        else 'normal',
                    # Italic
                    fontstyle='italic')


    for source, ax in zip((DBLP, APS, APS_CIT), a_ax):
        ax.set_xlabel("Decade",
                      labelpad=-.5)
        # Set source at top left
        ax.text(0.1, .95, str.upper(source), transform=ax.transAxes)

    # Show legend right to plot
    if not is_subplot:
        a_ax[0].set_ylabel('Distance')
        _ = fig.legend(
            loc='center left',
            bbox_to_anchor=(.975, 0.5),
            # Reduce line length
            # Reduce line width
            handlelength=1.25,
            labelspacing=0.25)
    return fig, a_ax, l_artists

In [ ]:
def plot_posterior_samples(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(
                ncols=len(L_DECADES_APS),
                nrows=2,
                sharex=True,
                sharey=True)

    a_hist = np.zeros(
        (2, len(L_DECADES_APS), len(BINS) - 1, len(BINS) - 1))
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    h = data['h']
                    tau = data['tau']
                    a_hist[i, j] = np.histogram2d(
                        h, tau, density=True, bins=BINS)[0]

    vmin, vmax = np.min(a_hist), np.max(a_hist)
    l_im = []
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            im = a_ax[i, j].imshow(
                a_hist[i, j],
                cmap=MAP_DATA_COLOR[source],
                origin='lower',
                extent=(0, 1, 0, 1),
                aspect='equal',
                vmin=vmin, vmax=vmax)
            a_ax[i, j].set_aspect('equal', adjustable='box')
            a_ax[i, j].axhline(
                .5,
                color='black',
                linestyle='--')
        l_im.append(im)

    return fig, a_ax, l_im

In [ ]:
def plot_posterior_gauss(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(
                ncols=len(L_DECADES_APS),
                nrows=2,
                sharex=True,
                sharey=True)

    a_gauss_h, a_gauss_tau = np.zeros(
        (2, 3, len(L_DECADES_APS), N_SAMPLES))
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=CompoundLFM.PAH.value,
                    lfm_tc=CompoundLFM.UNIFORM.value,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    a_gauss_h[i, j] = data['h']
                    a_gauss_tau[i, j] = data['tau']

    l_im = []
    for i, (source, l_decades) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT])):
        for j, decade in enumerate(l_decades):
            gauss_h = sp.stats.gaussian_kde(
                a_gauss_h[i, j])
            gauss_tau = sp.stats.gaussian_kde(
                a_gauss_tau[i, j])
            x = np.linspace(0, 1, 100)
            a_ax[j].plot(
                x, gauss_h(x), label=f"{source}", color=MAP_DATA_COLOR[source])
            a_ax[j].fill_between(
                x, gauss_h(x), alpha=.25, color=MAP_DATA_COLOR[source])
            a_ax[j].plot(
                x, -gauss_tau(x), color=MAP_DATA_COLOR[source])
            a_ax[j].fill_between(
                x, -gauss_tau(x), alpha=.25, color=MAP_DATA_COLOR[source])

    for ax in a_ax:
        # Set x-axis at y=0
        ax.spines['bottom'].set_position('zero')
        ax.set_xlim(0, 1)
        ax.set_xticks(
            [0, 0.5, 1],
            labels=["0", "", "1"])

    a_ax[0].spines['left'].set_position(('outward', 5))
    for ax in a_ax[1:]:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', labelleft=False, left=False)
        ax.set_ylabel('')

    a_ax[0].set_ylabel(
        'Density',
        # Locate at y=0,
        y=.65,
        # Bring closer to axis
        labelpad=-.5)
    yticks = a_ax[0].get_yticks()
    yticks = [2, 0, -2, -4, -6, -8, -10, -12]
    a_ax[0].set_yticks(
        yticks,
        labels=[f"{abs(x)}".format(x) for x in yticks])

    # Add tau and h labels
    a_ax[0].text(
        .1, -3, "$\\tau$",
        verticalalignment='center')
    a_ax[0].text(
        .1, 2, "$h$",
        verticalalignment='center')

    return fig, a_ax, l_im

In [ ]:
def plot_posterior_overview(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots()

    a_post_h_aps, a_post_tau_aps = np.zeros((2, len(L_DECADES_APS), 3))
    a_post_h_dblp, a_post_tau_dblp = np.zeros((2, len(L_DECADES_DBLP), 3))
    a_post_h_aps_cit, a_post_tau_aps_cit = np.zeros((2, len(L_DECADES_APS_CIT), 3))
    for i, (source, l_decades, a_post_h, a_post_tau) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT],
            [a_post_h_dblp, a_post_h_aps, a_post_h_aps_cit],
            [a_post_tau_dblp, a_post_tau_aps, a_post_tau_aps_cit])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    h = data['h']
                    tau = data['tau']
                    a_post_h[j] = np.quantile(h, q=(0.05, 0.5, 0.95))
                    a_post_tau[j] = np.quantile(tau, q=(0.05, 0.5, 0.95))

    for i, (source, l_decades, a_post_h, a_post_tau) in enumerate(zip(
            [DBLP, APS, APS_CIT],
            [L_DECADES_DBLP, L_DECADES_APS, L_DECADES_APS_CIT],
            [a_post_h_dblp, a_post_h_aps, a_post_h_aps_cit],
            [a_post_tau_dblp, a_post_tau_aps, a_post_tau_aps_cit])):
        for j, decade in enumerate(l_decades):
            ax.errorbar(
                l_decades,
                a_post_h[:, 1],
                xerr=a_post_h[:, 1] - a_post_h[:, 0],
                yerr=a_post_h[:, 2] - a_post_h[:, 1],
                fmt='s-',
                color=MAP_DATA_COLOR[source],
                alpha=.5)
            ax.errorbar(
                l_decades,
                a_post_tau[:, 1],
                xerr=a_post_tau[:, 1] - a_post_tau[:, 0],
                yerr=a_post_tau[:, 2] - a_post_tau[:, 1],
                fmt='^-',
                color=MAP_DATA_COLOR[source])

    return fig, ax

### File structure

In [ ]:
np_file.files

In [ ]:
np_file['f_m']

In [ ]:
np_file['h'].shape


In [ ]:
np_file['ccf']

### Other

In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]*(.9)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=3,
    height_ratios=[1, 1, 1]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=4,
    width_ratios=[1/6, 1/3, 1/3, 1/6],)

# Plot model selection
ax_model_selection_dblp = fig.add_subplot(
    gs_model_select[1])

ax_model_selection_aps = fig.add_subplot(
    gs_model_select[2],
    sharex=ax_model_selection_dblp,
    sharey=ax_model_selection_dblp)
_ = plot_model_selection(
    tpl_fig_ax=(fig, [ax_model_selection_dblp, ax_model_selection_aps]))

ax_model_selection_dblp.set_ylabel(
    'Distance')
ax_model_selection_aps.tick_params(axis='y', labelleft=False)

# Plot posterior samples
a_ax_posterior = [[], []]

# DBLP
gs_posterior_dblp = gs[1].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, .05, 1])
a_ax_posterior[0].append(fig.add_subplot(
    gs_posterior_dblp[0]))
for i, decade in enumerate(L_DECADES_DBLP[1:], start=1):
    a_ax_posterior[0].append(fig.add_subplot(
        gs_posterior_dblp[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))
# Add empty plot to DBLP
a_ax_posterior[0].append(None)

# APS
gs_posterior_aps = gs[2].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, 1, .05])
a_ax_posterior[1].append(fig.add_subplot(
    gs_posterior_aps[0],
    sharex=a_ax_posterior[0][0],
    sharey=a_ax_posterior[0][0]))
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_posterior[1].append(fig.add_subplot(
        gs_posterior_aps[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))

a_ax_posterior = np.asarray(a_ax_posterior)
_, _, l_im = plot_posterior_samples(
    tpl_fig_ax=(
        fig, a_ax_posterior))

for ax in a_ax_posterior[:, 0]:
    ax.set_ylabel(
        r"$h$",
        labelpad=-5)
    ax.set_yticks(
        [0, 1])
for ax in a_ax_posterior[-1].flatten():
    if ax is None:
        continue
    ax.set_xlabel(r"$\tau$",
        # Reduce distance to axis
        labelpad=-5)
    ax.set_xticks(
        [0, 1])

for i, decade in enumerate(L_DECADES_DBLP):
    _t = a_ax_posterior[0, i].set_title(
        f"{decade}",
        fontsize=10,
        pad=-.5)
# Add another year to the right of the last title at same height
fig.text(
    _t.get_position()[0] + 1.3,
    _t.get_position()[1] + .027,
    L_DECADES_APS[-1],
    transform=a_ax_posterior[0, -2].transAxes,
)

# Hide y-axis labels for all but the first column
for ax in a_ax_posterior[:, 1:].flatten():
    if ax is None:
        continue
    ax.tick_params(axis='y', labelleft=False)
    ax.set_ylabel('')

# Add colorbars
ax_cb_dblp = fig.add_subplot(
    gs_posterior_dblp[-2])
cbar_dblp = fig.colorbar(
    l_im[0],
    cax=ax_cb_dblp,
    orientation='vertical',)
cbar_dblp.set_label('PDE', loc='top', rotation=0, labelpad=8.5)

ax_cb_aps = fig.add_subplot(
    gs_posterior_aps[-1])
cbar_aps = fig.colorbar(
    l_im[1],
    cax=ax_cb_aps,
    orientation='vertical')
cbar_aps.set_label('PDE', loc='top', rotation=0, labelpad=8.5)

fig.text(0.02, 0.98, "a", fontweight='bold')
fig.text(0.02, 0.67, "b", fontweight='bold')

fig.savefig(os.path.join(
    folder_plots, "inference.pdf"), bbox_inches='tight')


In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]*(0.9)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=3,
    height_ratios=[1, 1, 1]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=2)

# Plot model selection
ax_model_selection_dblp = fig.add_subplot(
    gs_model_select[0])

ax_model_selection_aps = fig.add_subplot(
    gs_model_select[1],
    sharex=ax_model_selection_dblp,
    sharey=ax_model_selection_dblp)
_ = plot_model_selection(
    tpl_fig_ax=(fig, [ax_model_selection_dblp, ax_model_selection_aps]))

ax_model_selection_dblp.set_ylabel(
    'Distance')
ax_model_selection_aps.tick_params(axis='y', labelleft=False)

# Set axis and label to the right side
# ax_model_selection_dblp.yaxis.tick_right()
# ax_model_selection_dblp.yaxis.set_label_position("right")
# ax_model_selection_dblp.spines['right'].set_visible(True)
# ax_model_selection_dblp.spines['left'].set_visible(False)

# Hide y-axis
# ax_model_selection_aps.spines['left'].set_visible(False)
# ax_model_selection_aps.spines['right'].set_visible(True)
# ax_model_selection_aps.yaxis.tick_right()
# # Remove tick labels on y-axis for APS
# # ax_model_selection_aps.set_yticklabels([])
# # Remove y-axis label
# ax_model_selection_aps.set_ylabel('')

# Plot overview
# ax_overview = fig.add_subplot(
#     gs_model_select[-1],
#     sharex=ax_model_selection_dblp)
# ax_overview.set_xlabel(
#     "Decade",
#     labelpad=-.5)
# plot_posterior_overview(
#     tpl_fig_ax=(fig, ax_overview))


# ax_model_selection_legend = fig.add_subplot(
#     gs[0,
#        _x[0][4]:_x[0][5]])

# legend = ax_model_selection_legend.legend(
#     l_artists,
#     [f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$" for lfm_global, lfm_tc in LFM_COMBS],
#     loc='center',
#     # bbox_to_anchor=(0.975, 0.5),
#     # Reduce line length
#     # Reduce line width
#     handlelength=1.125,
#     labelspacing=0.125)
# legend.set_title("Model")
# ax_model_selection_legend.axis('off')

# Plot posterior samples
a_ax_posterior = [[], []]

# DBLP
gs_posterior_dblp = gs[1].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, .05, 1])
a_ax_posterior[0].append(fig.add_subplot(
    gs_posterior_dblp[0]))
for i, decade in enumerate(L_DECADES_DBLP[1:], start=1):
    a_ax_posterior[0].append(fig.add_subplot(
        gs_posterior_dblp[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))
# Add empty plot to DBLP
a_ax_posterior[0].append(None)

# APS
gs_posterior_aps = gs[2].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, 1, .05])
a_ax_posterior[1].append(fig.add_subplot(
    gs_posterior_aps[0],
    sharex=a_ax_posterior[0][0],
    sharey=a_ax_posterior[0][0]))
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_posterior[1].append(fig.add_subplot(
        gs_posterior_aps[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))

a_ax_posterior = np.asarray(a_ax_posterior)
_, _, l_im = plot_posterior_samples(
    tpl_fig_ax=(
        fig, a_ax_posterior))

# a_ax_posterior[1][-1].plot(
#     L_DECADES_APS,
#     np.zeros(len(L_DECADES_APS)),
# )

# Hide x-axis labels for all but the last row
# for ax in a_ax_posterior.flatten():
#     if ax is None: continue
#     ax.tick_params(axis='x', labelbottom=False)
#     ax.set_xticks(
#         [0, 1]
#     )
#     ax.set_xlabel('')
# a_ax_posterior[0, -1].set_xlabel(
#     r"$\tau$",
#     labelpad=-5)

for ax in a_ax_posterior[:, 0]:
    ax.set_ylabel(
        r"$h$",
        labelpad=-5)
    ax.set_yticks(
        [0, 1])
for ax in a_ax_posterior[-1].flatten():
    if ax is None:
        continue
    ax.set_xlabel(r"$\tau$",
        # Reduce distance to axis
        labelpad=-5)
    ax.set_xticks(
        [0, 1])

for i, decade in enumerate(L_DECADES_DBLP):
    _t = a_ax_posterior[0, i].set_title(
        f"{decade}",
        fontsize=10,
        pad=-.5)
# Add another year to the right of the last title at same height
fig.text(
    _t.get_position()[0] + 1.3,
    _t.get_position()[1] + .027,
    L_DECADES_APS[-1],
    transform=a_ax_posterior[0, -2].transAxes,
)

# Hide y-axis labels for all but the first column
for ax in a_ax_posterior[:, 1:].flatten():
    if ax is None:
        continue
    ax.tick_params(axis='y', labelleft=False)
    ax.set_ylabel('')

# Remove axis for last plot
# a_ax_posterior[1, -1].axis('off')

# Add colorbars
ax_cb_dblp = fig.add_subplot(
    gs_posterior_dblp[-2])
cbar_dblp = fig.colorbar(
    l_im[0],
    cax=ax_cb_dblp,
    orientation='vertical',)

ax_cb_aps = fig.add_subplot(
    gs_posterior_aps[-1])
cbar_aps = fig.colorbar(
    l_im[1],
    cax=ax_cb_aps,
    orientation='vertical')
cbar_aps.set_label('PDE', loc='top', rotation=0, labelpad=7.5)


# Adjust layout manually
# fig.subplots_adjust(
#     hspace=.35,
#     wspace=.25)

fig.text(0.02, 0.98, "a", fontweight='bold')
fig.text(0.02, 0.67, "b", fontweight='bold')
# fig.text(0.666, 0.98, "c", fontweight='bold')

# fig.tight_layout()
fig.savefig(os.path.join(
    folder_plots, "inference.pdf"), bbox_inches='tight')


## Summary statistics

In [ ]:
FOLDER_ARRAY_POOL = "summary_stats_sim"
# L_METRICS = ["ccf_0", "ccf_1", "ccf_2", "ccf_3", "ccf_4", "ccf_5", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]

# L_METRICS = ["ccf_0", "ccf_3", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]

In [ ]:
def load_sim_metrics(
        source: str,
        lfm_global: str,
        decade: int,
        lfm_tc: str) -> Dict[str, np.ndarray]:
    """Load metrics from the array pool."""
    folder_name = create_folder_name(
        source=source,
        decade=decade,
        lfm_global=lfm_global,
        lfm_tc=lfm_tc)
    return {
        metric: np.load(
            os.path.join(folder_name, FOLDER_ARRAY_POOL, f"{metric}.npy"))
        for metric in L_METRICS
    }

In [ ]:
def load_obs_metric(
        source: str,
        decade: int,
        lfm_global: str,
        lfm_tc: str,
):
    return {metric: np.load(
        create_posterior_file_name(
            create_folder_name(
                source=source,
                lfm_global=lfm_global,
                lfm_tc=lfm_tc,
                decade=decade)))[metric]\
                    for metric in L_METRICS}

In [ ]:
from collections import defaultdict

f, a_ax = plt.subplots(
    3, len(L_DECADES_APS),
    figsize=(SIZE_FIG[0], SIZE_FIG[1]),
    layout='constrained',
    sharey="row",
    sharex=True,)

for a_ax_src, source, l_decades in zip(
        a_ax, [APS, APS_CIT, DBLP], [L_DECADES_APS, L_DECADES_APS_CIT, L_DECADES_DBLP]):
    for decade, ax in zip(
            l_decades, a_ax_src):

        d_metrics = load_sim_metrics(
            source=source,
            decade=decade,
            lfm_global=CompoundLFM.PAH.value,
            lfm_tc=CompoundLFM.UNIFORM.value)
        d_metrics_obs = load_obs_metric(
            source=source,
            decade=decade,
            lfm_global=CompoundLFM.PAH.value,
            lfm_tc=CompoundLFM.UNIFORM.value)

        n_iter = d_metrics["ccf"].shape[0] // 1000
        d_metric_stds = defaultdict(list)

        for i, metric in enumerate(L_METRICS):
            for j in range(n_iter):
                # Reversed because we want the last 1000 iterations
                data = d_metrics[metric][-(j + 1) * 1000:-(j * 1000)\
                                        if j > 0 else None]
                d_metric_stds[metric].append(
                    np.sqrt(np.mean((data - d_metrics_obs[metric])**2)))
            ax.plot(
                range(1, n_iter + 1),
                d_metric_stds[metric][::-1],
                label=metric if ((decade == L_DECADES_APS[0]) and (source == APS)) else None,
                marker='None',
                # markersize=3,
                # linewidth=.75,
                alpha=.75,
                color=plt.colormaps.get_cmap("tab10")(i)
            )
            # ax.axhline(
            #     d_metrics_obs[metric],
            #     color=plt.colormaps.get_cmap("tab10")(i)
            #     )
# a_ax[0].legend()
for ax in a_ax[:, 0]:
    ax.set_ylabel("Distance", labelpad=-.5)
for ax, decade in zip((a_ax[0, :]),
                      L_DECADES_APS):
    ax.set_title(f"{decade}", fontsize=10, pad=-.5)
a_ax[-1, 2].set_xlabel("Iteration (1000 samples)")
for source, pos_y in zip(
        [APS, APS_CIT, DBLP],
        np.linspace(1, 0.4, 3)):
    f.text(
        0.0, pos_y,
        source,
        fontweight='bold',
        horizontalalignment='left',
        verticalalignment='top',)
f.legend(bbox_to_anchor=(0.5, 1.),
         loc='lower center',
         ncol=len(L_METRICS),
         handlelength=.75,
         labelspacing=0.125,
         borderpad=0.25,
         handletextpad=0.25,)


In [ ]:
d_metrics["ccf"][-1000:].mean(), d_metrics["ccf"][-1000:].std()

In [ ]:
d_metrics["ccf"][:1000].mean(), d_metrics["ccf"][:1000].std()

In [ ]:
d_metrics["mann_whitney"].shape

In [ ]:
a_posteriors = np.load(
    os.path.join(create_folder_name(
        lfm_global=CompoundLFM.PAH.value,
        lfm_tc=CompoundLFM.PAH.value), "posteriors.npz")
)
print("Observed metrics:")
for metric in L_METRICS:
    print(f"{metric}_obs: {a_posteriors[metric]}")

In [ ]:
print("Mean and std of simulations:")
for metric, data in d_metrics.items():
    print(f"{metric}: {np.mean(data)}, {np.std(data)}")

In [ ]:
# Plot metric distributions in a grid
fig, axes = plt.subplots(
    3, 5, figsize=(20, 12))
for i, metric in enumerate(L_METRICS):
    ax = axes.flatten()[i]
    data = d_metrics[metric]
    ax.hist(data, bins=25, density=True)
    ax.axvline(a_posteriors[metric], color='r', linestyle='--')
    ax.set_title(metric)
fig.tight_layout()